In [1]:
!pip install av

In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import r3d_18, R3D_18_Weights
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize, RandomHorizontalFlip, ColorJitter
import av
import numpy as np
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
import torch.optim as optim
from itertools import cycle
from sklearn.metrics import f1_score, precision_score, recall_score

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 8
NUM_CLASSES = 11
LEARNING_RATE = 1e-5  # Lowered learning rate for stability
EPOCHS = 50
ALPHA = 0.99  # EMA decay rate for teacher updates
CONFIDENCE_THRESHOLD = 0.85  # Lowered confidence threshold for pseudo-labels
CLIP_LEN = 16  # Number of frames per clip
TEMPERATURE = 2.0  # For smoothing pseudo-labels

In [3]:
class VideoDataset(Dataset):
    def __init__(self, video_files, labels, clip_len=16, is_unlabeled=False):
        self.video_files = video_files
        self.labels = labels
        self.clip_len = clip_len
        self.is_unlabeled = is_unlabeled
        self.label_encoder = LabelEncoder()

        if not is_unlabeled:
            self.label_encoder.fit(labels)

        self.transform = Compose([
            Resize((128, 128)),
            RandomHorizontalFlip(p=0.5),
            ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            CenterCrop(112),
            ToTensor(),
            Normalize(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225]),
        ])

    def __len__(self):
        return len(self.video_files)

    def sample_frame_indices(self, seg_len):
        return np.linspace(0, seg_len - 1, self.clip_len, dtype=np.int64)

    def read_video(self, file_path, indices):
        container = av.open(file_path)
        frames = []
        for i, frame in enumerate(container.decode(video=0)):
            if i > indices[-1]:
                break
            if i in indices:
                frame = frame.to_image()
                frames.append(self.transform(frame))
        video_tensor = torch.stack(frames).permute(1, 0, 2, 3)  # [channels, frames, height, width]
        return video_tensor

    def __getitem__(self, idx):
        video_file = self.video_files[idx]
        container = av.open(video_file)
        seg_len = container.streams.video[0].frames
        indices = self.sample_frame_indices(seg_len)

        video = self.read_video(video_file, indices)

        if not self.is_unlabeled:
            label = self.labels[idx]
            label = self.label_encoder.transform([label])[0]
            return {'video': video, 'label': label}
        else:
            return {'video': video}

In [4]:
def load_data(folder, label_map, is_unlabeled=False):
    videos = []
    labels = []

    files = os.listdir(folder)
    for file in files:
        if file == '.ipynb_checkpoints':
            continue
        else:
            if is_unlabeled:
                videos.append(os.path.join(folder, file))
            else:
                label = file.split('_')[1]
                label = label_map[label]
                videos.append(os.path.join(folder, file))
                labels.append(label)

    if is_unlabeled:
        return VideoDataset(videos, labels=None, is_unlabeled=True)
    return VideoDataset(videos, labels)

In [5]:
def initialize_resnet3d_model(num_classes=11, pretrained=True):
    model = r3d_18(weights=R3D_18_Weights.DEFAULT if pretrained else None)
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 512),  # Intermediate layer
        nn.ReLU(),
        nn.Dropout(p=0.4),  # Dropout for regularization
        nn.Linear(512, num_classes)
    )
    return model.to(DEVICE)

In [6]:
def update_teacher(student_model, teacher_model, alpha=ALPHA):
    for teacher_param, student_param in zip(teacher_model.parameters(), student_model.parameters()):
        teacher_param.data = alpha * teacher_param.data + (1 - alpha) * student_param.data

In [7]:
def train_with_kl_divergence(student_model, teacher_model, train_loader, unlabeled_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    kl_loss_fn = nn.KLDivLoss(reduction="batchmean")
    optimizer = optim.AdamW(student_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

    for epoch in range(EPOCHS):
        student_model.train()
        total_loss = 0

        labeled_cycle = cycle(train_loader)
        for unlabeled_batch in unlabeled_loader:
            labeled_batch = next(labeled_cycle)

            # Labeled data
            labeled_videos = labeled_batch['video'].to(DEVICE)
            labels = labeled_batch['label'].to(DEVICE)

            # Unlabeled data
            unlabeled_videos = unlabeled_batch['video'].to(DEVICE)

            # Supervised loss
            labeled_outputs = student_model(labeled_videos)
            loss_supervised = criterion(labeled_outputs, labels)

            # Consistency loss with pseudo-labels
            with torch.no_grad():
                teacher_outputs = teacher_model(unlabeled_videos)
                pseudo_labels = torch.softmax(teacher_outputs / TEMPERATURE, dim=1)
                confidence_mask = pseudo_labels.max(dim=1)[0] > CONFIDENCE_THRESHOLD

            student_outputs = student_model(unlabeled_videos)
            student_predictions = torch.log_softmax(student_outputs, dim=1)

            if confidence_mask.any():
                loss_consistency = kl_loss_fn(
                    student_predictions[confidence_mask], pseudo_labels[confidence_mask]
                )
            else:
                loss_consistency = 0

            # Total loss
            loss_total = loss_supervised + 0.5 * loss_consistency
            optimizer.zero_grad()
            loss_total.backward()
            optimizer.step()

            # Update teacher model
            update_teacher(student_model, teacher_model)
            total_loss += loss_total.item()

        scheduler.step()  # Update learning rate

        print(f"Epoch [{epoch + 1}/{EPOCHS}], Loss: {total_loss:.4f}")
        evaluate(student_model, val_loader)


In [8]:
def evaluate(model, val_loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            videos = batch['video'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            outputs = model(videos)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100 * (sum(1 for p, l in zip(all_preds, all_labels) if p == l) / len(all_labels))
    f1 = f1_score(all_labels, all_preds, average='weighted')
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')

    print(f"Validation Accuracy: {accuracy:.2f}%")
    print(f"F1 Score: {f1:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")

    # Class-wise metrics
    class_f1 = f1_score(all_labels, all_preds, average=None)
    print("Class-wise F1 Scores:", class_f1)

In [9]:
label_map = {'Arm': 0, 'bs': 1, 'ce': 2, 'sq': 3, 'dr': 4, 'mfs': 5, 'ms': 6, 'sac': 7, 'fg': 8, 'tr': 9, 'tw': 10}
train_folder = 'New_Data/Train_Data_64_Orginal'
validate_folder = 'New_Data/Validation_Video_64'
unlabeled_folder = 'New_Data/Unlabled_Data_64'



In [10]:
# Load datasets
train_dataset = load_data(train_folder, label_map)
val_dataset = load_data(validate_folder, label_map)
unlabeled_dataset = load_data(unlabeled_folder, label_map, is_unlabeled=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=True)


In [11]:
student_model = initialize_resnet3d_model(NUM_CLASSES)
teacher_model = initialize_resnet3d_model(NUM_CLASSES)
teacher_model.load_state_dict(student_model.state_dict())  # Sync student and teacher


<All keys matched successfully>

In [ ]:
train_with_kl_divergence(student_model, teacher_model, train_loader, unlabeled_loader, val_loader)